# shillelagh

## Example 1: SQL over a CSV file

In [ ]:
# Example 1: the csvfile adapter exposes a CSV file as a table SQLite can filter and aggregate
import csv, pathlib, tempfile
from shillelagh.backends.apsw.db import connect

path = pathlib.Path(tempfile.mkdtemp()) / "sales.csv"
with path.open("w", newline="") as handle:
    writer = csv.writer(handle, quoting=csv.QUOTE_NONNUMERIC)  # the adapter reads quoted text, bare numbers
    writer.writerows([["region", "units"], ["north", 10], ["south", 4], ["north", 7]])
connection = connect(":memory:", adapters=["csvfile"])
rows = connection.cursor().execute(
    f'SELECT region, SUM(units) FROM "{path}" GROUP BY region ORDER BY region').fetchall()
assert rows == [("north", 17.0), ("south", 4.0)], rows
print("Example 1:", rows)

## Example 2: SQL over a pandas DataFrame

In [ ]:
# Example 2: the pandasmemory adapter queries a DataFrame by its variable name
import pandas as pd
from shillelagh.backends.apsw.db import connect

prices = pd.DataFrame({"ticker": ["A", "B", "C"], "price": [10.5, 99.0, 42.0]})
connection = connect(":memory:", adapters=["pandasmemory"])
rows = connection.cursor().execute("SELECT ticker FROM prices WHERE price > 20 ORDER BY price").fetchall()
assert rows == [("C",), ("B",)], rows
print("Example 2:", rows)

## Example 3: the SQLAlchemy dialect

In [ ]:
# Example 3: create_engine("shillelagh://") gives SQLAlchemy access to the same adapters
import csv, pathlib, tempfile
from sqlalchemy import create_engine, text

path = pathlib.Path(tempfile.mkdtemp()) / "people.csv"
with path.open("w", newline="") as handle:
    csv.writer(handle, quoting=csv.QUOTE_NONNUMERIC).writerows([["name", "age"], ["ada", 36], ["alan", 41]])
engine = create_engine("shillelagh://", adapters=["csvfile"])
with engine.connect() as connection:
    rows = connection.execute(text(f'SELECT name FROM "{path}" WHERE age > 40')).fetchall()
assert [tuple(r) for r in rows] == [("alan",)], rows
print("Example 3:", engine.dialect.name, rows)

## Example 4: generated rows joined with a real table

In [ ]:
# Example 4: the virtualmemory adapter generates rows; SQLite joins them with a DataFrame
import pandas as pd
from shillelagh.backends.apsw.db import connect

weekdays = pd.DataFrame({"n": [0, 1, 2], "name": ["mon", "tue", "wed"]})
connection = connect(":memory:", adapters=["virtualmemory", "pandasmemory"])
rows = connection.cursor().execute(
    'SELECT v.int, w.name FROM "virtual://?cols=int:int&rows=5" AS v JOIN weekdays AS w ON v.int = w.n').fetchall()
assert rows == [(0, "mon"), (1, "tue"), (2, "wed")], rows
print("Example 4:", rows)

## Example 5: this machine's CPU usage as a table

In [ ]:
# Example 5: the systemapi adapter (psutil) reports CPU usage, one row per sample
from shillelagh.backends.apsw.db import connect

connection = connect(":memory:", adapters=["systemapi"])
cursor = connection.cursor()
rows = cursor.execute('SELECT * FROM "system://cpu?interval=0.1" LIMIT 1').fetchall()
columns = [column[0] for column in cursor.description]
assert len(rows) == 1 and columns[0] == "timestamp" and len(columns) > 1, (columns, rows)
assert all(0.0 <= value <= 100.0 for value in rows[0][1:]), rows
print("Example 5:", len(columns) - 1, "CPU columns; first:", columns[:3])